<a href="https://colab.research.google.com/github/maandressa/databricks-pyspark/blob/main/PySpark1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Criando uma seção
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, asc, desc, when, sum

spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [ ]:
#Importando um arquivo CSV e especificando um Header no arquivo
import requests
url = "https://raw.githubusercontent.com/ashaypatil11/spark/main/operations_management.csv"
r = requests.get(url)

with open("/tmp/operations_management.csv", "wb") as f:
     f.write(r.content)

# Agora lê com Spark
data = spark.read.format("csv") \
     .option("header", "true") \
     .option("inferSchema", "true") \
     .load("/tmp/operations_management.csv")
data.show()
data.printSchema()
data.describe().show()

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-4690995288634839>, line 14
      9 # Agora lê com Spark 
     10 data = spark.read.format("csv") \
     11      .option("header", "true") \
     12      .option("inferSchema", "true") \
     13      .load("/tmp/operations_management.csv") 
---> 14 data.show() 
     15 data.printSchema() 
     16 data.describe().show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1123, in DataFrame.show(self, n, truncate, vertical)
   1122 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1123     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:876, in DataFrame._show_string(self, n, truncate, vertical)
    859     except ValueError:
    860         raise PySparkTyp

In [ ]:
print(spark.version)

4.0.0


In [ ]:
data = [('Andressa', '29'), ('Bruno', '34'), ('Arthur', '9'), ('Alice', '6')]
columns = ['name', 'age']
df = spark.createDataFrame(data, columns)

In [ ]:
df.show()

+--------+---+
|    name|age|
+--------+---+
|Andressa| 29|
|   Bruno| 34|
|  Arthur|  9|
|   Alice|  6|
+--------+---+



In [ ]:
df.display()

name,age
Andressa,29
Bruno,34
Arthur,9
Alice,6


In [ ]:
#Filtrar dados com .filter()

df_filter = df.filter(col('age') > 10)
df_filter.show()

+--------+---+
|    name|age|
+--------+---+
|Andressa| 29|
|   Bruno| 34|
+--------+---+



In [ ]:
#Ordenar Dados com OrderBy()

df_order = df.orderBy(desc("age"))
df_order.display()


name,age
Arthur,9
Alice,6
Bruno,34
Andressa,29


In [ ]:
#Criar uma nova coluna no DataFrame
df_NewColumn = df.withColumn('Idade depois de 5 anos', col("age")+5)
df_MoreColumns = df_NewColumn.withColumn('Idade depois de 10 anos', col('age')+10)
df_MoreColumns.display()

name,age,Idade depois de 5 anos,Idade depois de 10 anos
Andressa,29,34,39
Bruno,34,39,44
Arthur,9,14,19
Alice,6,11,16


In [ ]:
#Transformações e Ações

df_transform = df.select("name", "age").filter(df.age > 10)
df_transform.display()
#

name,age
Andressa,29
Bruno,34


In [ ]:
#Manipulando Dados
dados = [('Andressa', 'AOS', '4000'),
         ('Pamella', 'SRE', '5500'),
         ('Aldo', 'AOC', '4500'),
         ('Dion', 'AOS', '3500')]
colunas = ['Name', 'Dept', 'Sal']

df_new = spark.createDataFrame(dados, colunas)
df_new.display()

Name,Dept,Sal
Andressa,AOS,4000
Pamella,SRE,5500
Aldo,AOC,4500
Dion,AOS,3500


In [ ]:
df_new.describe()

DataFrame[summary: string, Name: string, Dept: string, Sal: string]

In [ ]:
#Fitrar pelo departamento
df_sales = df_new.filter(df_new['Dept']=='AOS')
df_sales.display()

Name,Dept,Sal
Andressa,AOS,4000
Dion,AOS,3500


In [ ]:
#Converte para o tipo numerico a coluna 'Sal'

df_new2 = df_new.withColumn("Sal", col("Sal").cast("double"))
df_new2.display()

Name,Dept,Sal
Andressa,AOS,4000.0
Pamella,SRE,5500.0
Aldo,AOC,4500.0
Dion,AOS,3500.0


In [ ]:
#Fazer uma media de salario por departamento

media_sal = df_new2.groupBy('Dept').avg('Sal')
media_sal.display()

Dept,avg(Sal)
AOS,3750.0
SRE,5500.0
AOC,4500.0


In [ ]:
#Criando nova coluna refrenciando outra com condições

df_new2.withColumn('Categoria Salario', when(col('Sal') < 4001, 'Baixo').otherwise('Alto')).display()

Name,Dept,Sal,Categoria Salario
Andressa,AOS,4000.0,Baixo
Pamella,SRE,5500.0,Alto
Aldo,AOC,4500.0,Alto
Dion,AOS,3500.0,Baixo


In [ ]:
#Agrupar por setor e somar os valores pela coluna 'Sal'

df_new2.groupBy("Dept").agg(sum("Sal").alias('Total por Setor')).display()

Dept,Total por Setor
AOS,7500.0
SRE,5500.0
AOC,4500.0
